# Preparación de Datos - Train/Test Split

**Proyecto:** Sistema de Clasificación de Acciones S&P 500

**Grupo 27** - Universidad de Los Andes

---

## Objetivo

Preparar los datos para modelado:
- Cargar dataset consolidado
- Split temporal train/test (80/20)
- Guardar archivos separados para reutilizar en todos los experimentos

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime

## 1. Carga de Datos

In [4]:
# Cargar dataset completo
df = pd.read_parquet('../../data/processed/ml_ready/features_combined.parquet')

print(f"Dataset cargado: {df.shape[0]:,} filas, {df.shape[1]} columnas")
df.head()

Dataset cargado: 25,160 filas, 20 columnas


,Ticker,Date,SMA_20,SMA_50,EMA_12,EMA_26,RSI_14,MACD,MACD_signal,MACD_diff,BB_upper,BB_middle,BB_lower,BB_width,OBV,ATR_14,Returns,Volatility_10,Volume_change,Target
0,AAPL,2015-01-02 00:00:00-05:00,24.817478,24.751199,24.793362,24.825706,42.667413,-0.032344,0.032304,-0.064648,25.801741,24.817478,23.833214,0.079320,219697360800,0.547673,-0.009512,0.014881,0.285030,0
1,AAPL,2015-01-05 00:00:00-05:00,24.714957,24.767641,24.606318,24.733252,36.165196,-0.126934,0.000456,-0.127390,25.765108,24.714957,23.664806,0.084981,219440218800,0.570687,-0.028172,0.013245,0.208270,1
2,AAPL,2015-01-06 00:00:00-05:00,24.617983,24.775996,24.448390,24.647810,36.199033,-0.199420,-0.039519,-0.159901,25.710478,24.617983,23.525489,0.088756,219703407200,0.574305,0.000094,0.013346,0.023514,1
3,AAPL,2015-01-07 00:00:00-05:00,24.566390,24.789240,24.365628,24.593190,41.222076,-0.227562,-0.077127,-0.150434,25.689771,24.566390,23.443010,0.091457,219863830800,0.564034,0.014023,0.013852,-0.390461,1
4,AAPL,2015-01-08 00:00:00-05:00,24.541648,24.821345,24.436936,24.610667,52.428622,-0.173731,-0.096448,-0.077283,25.618008,24.541648,23.465287,0.087717,220101288800,0.593488,0.038422,0.019440,0.480194,1


In [5]:
# Verificar estructura
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25160 entries, 0 to 25159
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype                           
---  ------         --------------  -----                           
 0   Ticker         25160 non-null  object                          
 1   Date           25160 non-null  datetime64[ns, America/New_York]
 2   SMA_20         25160 non-null  float64                         
 3   SMA_50         25160 non-null  float64                         
 4   EMA_12         25160 non-null  float64                         
 5   EMA_26         25160 non-null  float64                         
 6   RSI_14         25160 non-null  float64                         
 7   MACD           25160 non-null  float64                         
 8   MACD_signal    25160 non-null  float64                         
 9   MACD_diff      25160 non-null  float64                         
 10  BB_upper       25160 non-null  float64                    

In [6]:
# Distribución de Target
print("\nDistribución de Target:")
print(df['Target'].value_counts())
print(f"\nBalance: {df['Target'].value_counts(normalize=True) * 100}")


Distribución de Target:
Target
1    13281
0    11879
Name: count, dtype: int64

Balance: Target
1    52.786169
0    47.213831
Name: proportion, dtype: float64


## 2. Train/Test Split Temporal

Para datos financieros es crítico usar split temporal:
- Entrenar con datos pasados
- Evaluar en datos futuros (simulando producción)

**Split:** 80% train / 20% test

In [7]:
# Ordenar por fecha para asegurar split temporal correcto
df_sorted = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# Calcular índice de corte (80%)
split_idx = int(len(df_sorted) * 0.8)

# Separar train y test
train = df_sorted.iloc[:split_idx].copy()
test = df_sorted.iloc[split_idx:].copy()

print(f"Train set: {len(train):,} filas ({len(train)/len(df_sorted)*100:.1f}%)")
print(f"Test set:  {len(test):,} filas ({len(test)/len(df_sorted)*100:.1f}%)")

Train set: 20,128 filas (80.0%)
Test set:  5,032 filas (20.0%)


In [8]:
# Verificar rangos de fechas
print("\nRangos de fechas:")
print(f"Train: {train['Date'].min()} a {train['Date'].max()}")
print(f"Test:  {test['Date'].min()} a {test['Date'].max()}")


Rangos de fechas:
Train: 2015-01-02 00:00:00-05:00 a 2024-12-31 00:00:00-05:00
Test:  2015-01-02 00:00:00-05:00 a 2024-12-31 00:00:00-05:00


In [9]:
# Verificar balance en ambos sets
print("\nBalance de Target en Train:")
print(train['Target'].value_counts(normalize=True) * 100)

print("\nBalance de Target en Test:")
print(test['Target'].value_counts(normalize=True) * 100)


Balance de Target en Train:
Target
1    53.07035
0    46.92965
Name: proportion, dtype: float64

Balance de Target en Test:
Target
1    51.649444
0    48.350556
Name: proportion, dtype: float64


## 3. Guardar Datasets

Guardamos train.parquet y test.parquet para reutilizar en todos los notebooks de modelado.

In [10]:
# Guardar datasets procesados
train.to_parquet('../../data/processed/ml_ready/train.parquet', index=False)
test.to_parquet('../../data/processed/ml_ready/test.parquet', index=False)

print("Datasets guardados:")
print("  data/processed/ml_ready/train.parquet")
print("  data/processed/ml_ready/test.parquet")

Datasets guardados:
  data/processed/ml_ready/train.parquet
  data/processed/ml_ready/test.parquet


## Resumen

- Dataset original: 25,160 observaciones
- Train: 20,128 observaciones (2015 a ~2023)
- Test: 5,032 observaciones (~2023 a 2024)
- Balance mantenido en ambos sets

**Próximos pasos:** Usar estos datasets en los notebooks de modelado.